**1. Klasifikatori:** Modeli koji na osnovu ulaznih podataka dodjeljuju podatke u unaprijed definisane kategorije ili klase.
Izlaz: diskretna klasa (npr. spam/nije spam, bolest/nije bolest).

**2. Regresori:** Modeli koji na osnovu ulaznih podataka predvi?aju numeri?ku, kontinuiranu vrednost.
Izlaz: realan broj (npr. cijena ku?e, temperatura).

**3. Random Forest:** Sastoji se od vi?e stabala odlu?ivanja (decision trees). Svako stablo daje svoju prognozu, a finalna odluka se donosi glasovanjem (naj?e??a klasa). Mo?e raditi i regresiju (tada je RandomForestRegressor) i klasifikaciju (RandomForestClassifier).

### HEART DISEASE DATASET - KNeighborsClassifier

Zadatak:
1. U?itavanje realnog skupa: Preuzeti Heart Disease dataset u CSV formatu: 
            https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data.

2. Vektorizacija kategorijskih podataka: target (atribut koji govori o prisustvu bolesti) kodirati sa LabelEncoder. Kategorijske kolone (cp, thal, slope) pretvoriti u dummy varijable pomo?u pd.get_dummies.

3. Imputacija: U?itani nedostaju?i podaci ozna?eni su sa ? ? zamijeniti ih sa NaN. Za numeri?ke kolone primijeniti strategiju 'mean', a za kategorijske 'most_frequent'.

4. Uklanjanje outliera: Primijeniti pravilo 3 standardne devijacije na kolonu chol (holesterol).

5. Skaliranje: Primijeniti StandardScaler ili MinMaxScaler na numeri?ke kolone.

6. Evaluacija: Podijeliti podatke na trening i test skup u odnosu 80%/20%. Trenirati KNeighborsClassifier. Ispisati metrike: accuracy, precision, recall ? prvo za sirove podatke (bez skaliranja i dummy-a), a zatim nakon pune obrade.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score

In [ ]:
# dataset je drugaciji od onih sa seaborn-a pa je potrebno navesti sva imena kolona koje ima
column_names = ['age','sex','cp','trestbps','chol','fbs','restecg',
                'thakach','exang','oldpeak','slope','ca','thal','num'] 

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
# ucitava dataset sa url adrese, kolonama dodjeljuje imena, ? koji se nalaze kao podaci zamijenice sa NaN, ukljanja space poslije , u csv fajlu
df = pd.read_csv(url, names=column_names, na_values=' ?',skipinitialspace=True)
df.head()

In [ ]:
# prvo se vrsi treniranje modela na neobradjenim podacima

# X su svi podaci osim target kolone koja je u ovom slucaju num - stepen napredovanja bolesti
X_raw = df.drop('num', axis=1).copy()

# posto imaju 0,1,2,3,4 klase u zavisnosti od stepena enkoduje 0 kao 0 - nema bolest, a 1,2,3,4 kao 1 - ima bolest
Y_raw = LabelEncoder().fit_transform(df['num']) # sadrzi samo target kolonu - num

In [ ]:
# podjela podataka za test i trening, 80/20
X_train_raw, X_test_raw, Y_train_raw, Y_test_raw = train_test_split(X_raw, Y_raw, test_size=0.2, random_state=42)

# razdvajanje kategorickih i numerickih kolona zbog lakse obrade kasnije
cat_cols = X_train_raw.select_dtypes(include=['object']).columns
num_cols = X_train_raw.select_dtypes(include=['int64', 'float64']).columns

In [ ]:
# posto se podaci jos uvijek nece u potpunosti obradjivati potrebno je NaN podatke u 
# numerickim kolonama zamijeniti nekom vrijednoscu - prosjecna u ovom slucaju 'mean'
# u kategorickim kolonama na tim mjestima upisuje se rucno 'Missing'


def prepare_data(df):
    df_copy = df.copy()
    num_imputer = SimpleImputer(strategy='mean')
    df_copy[num_cols] = num_imputer.fit_transform(df_copy[num_cols])

    for col in cat_cols:
        df_copy[col] = df_copy[col].fillna('Missing')
        df_copy[col] = LabelEncoder().fit_transform(df_copy[col])

    return df_copy

X_train_raw_prep = prepare_data(X_train_raw)
X_test_raw_prep = prepare_data(X_test_raw)

In [ ]:
# krece se sa treniranjem modela pomocu klasifikacije, bira se 5 najblizih susjeda u grafu
# model se uvijek trenira na osnovu X_train i Y_train podataka
# predvidjanje se vrsi na osnovu X_test, a mjerenje koliko je model dobar na osnovu poredjenja Y_pred i Y_test

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_raw_prep, Y_train_raw)

Y_pred_raw = knn.predict(X_test_raw_prep)

# average='macro' - ako nije navedeno funkcije podrazumijevaju da su kolone ispisane u binarnim podacima, ovo omogucava viseklasni rad
# zero_division=0 - ukoliko model nije uopste pogodio neke klase doci ce po formuli za preciznost do 
                    # dijeljenja sa nulom pa se to sprijecava s ovim argumentom

print("Raw data:\n")
print(f'Accuracy Score: {accuracy_score(Y_test_raw, Y_pred_raw):.2f}')
print(f"Precision Score: {precision_score(Y_test_raw, Y_pred_raw, average='macro', zero_division=0):.2f}")
print(f"Recall Score: {recall_score(Y_test_raw, Y_pred_raw, average='macro'):.2f}")

In [ ]:
# prelazak na obradu podataka, vrsi se kopija dataseta df
df_full = df.copy()
df_full.head()

In [ ]:
# numericke kolone se popunjavaju sa srednjom vrijednoscu, a kategoricke sa najcescom koja se ponavlja
num_imputer = SimpleImputer(strategy='mean')
df_full[num_cols] = num_imputer.fit_transform(df_full[num_cols])

cat_imputer = SimpleImputer(strategy='most_frequent')
df_full[cat_cols] = cat_imputer.fit_transform(df_full[cat_cols])

In [ ]:
# uklanjaju se outlejeri na osnovu kolone 'chol' racunanjem standardne devijacije

chol = df_full['chol']
mask = np.abs(chol - chol.mean()) < 3*chol.std() # uvijek ova formula za odredjivanje maske
df_full = df_full[mask] # zadrzavaju se samo one kolone za koje je mask = True, ostale se odbacuju (outlejeri)

# ponovo se traze kategoricke kolone jer je moguce da nakon obrade podataka vise kategoricke kolone nisu iste kao originalne
cat_cols = df_full.select_dtypes(include=['object']).columns 
# vrsi se one-hot enkodovanje - svaka kategorijska kolona se pretvara u vise binarnih kolona - kasnije se koristi OneHotEncoder
df_full = pd.get_dummies(df_full, columns=cat_cols, drop_first=True)

# vrsi skaliranje numericnih kolona tako da svaka kolona ima srednju vrijednost (0) i standardnu devijaciju (1)
scaler = StandardScaler()
df_full[num_cols] = scaler.fit_transform(df_full[num_cols])

In [ ]:
X = df_full.drop('num', axis=1).copy()
Y = LabelEncoder().fit_transform(df_full['num'])

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

knn = KNeighborsClassifier(n_neighbors=5)
knn = knn.fit(X_train, Y_train)

Y_pred = knn.predict(X_test)

print("After proccesiong data: ")
print(f'Accuracy Score: {accuracy_score(Y_test, Y_pred): .2f}')
print(f"Precision Score: {precision_score(Y_test, Y_pred, average='macro', zero_division=0): .2f}")
print(f"Recall Score: {recall_score(Y_test, Y_pred, average='macro'): .2f}")


### PENGUINS - Stabla odlucivanja

1. U?itavanje realnog skupa: U?itajte penguins skup podataka pomo?u biblioteke seaborn: https://github.com/allisonhorst/palmerpenguins

2. Vektorizacija kategorijskih podataka: Identifikujte kategorijske kolone (npr. island, sex) i primijenite OneHotEncoder ili LabelEncoder.

3. Imputacija: Detektujte NaN vrijednosti i zamijenite ih: Srednjom vrijedno??u za numeri?ke kolone. Naj?e??om vrijedno??u za kategorijske kolone.

4. Outlieri: Detektujte i uklonite outliere za kolonu body_mass_g na osnovu praga ?3 standardne devijacije.

5. Skaliranje: Normalizujte numeri?ke kolone bill_length_mm, bill_depth_mm, flipper_length_mm, body_mass_g pomo?u StandardScaler.

6. Trening: Podijelite podatke na trening/test skup. Trenirajte RandomForestClassifier za predikciju vrste pingvina (species kolona).

7. Evaluacija: Evaluirajte ta?nost, preciznost i confusion matrix modela na testnom skupu. Prika?ite klasifikacioni izvje?taj (classification_report).

In [ ]:
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [ ]:
df = sns.load_dataset('penguins')
df.head()

In [ ]:
cat_cols = df.select_dtypes(include=['object']).columns
num_cols = df.select_dtypes(include=['float64', 'int64']).columns
df[num_cols].isnull().sum()

In [ ]:
df_encoded = df.copy()
le_encoder = LabelEncoder()
for column in cat_cols:
    df_encoded[column] = le_encoder.fit_transform(df_encoded[column])

df_encoded.head()

In [ ]:
ohe_cat_cols = ['island', 'sex']
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded_data = ohe.fit_transform(df_encoded[ohe_cat_cols])
feature_names = ohe.get_feature_names_out(ohe_cat_cols)
df_ohe = pd.DataFrame(encoded_data, columns=feature_names, index = df.index)

df_encoded = df_encoded.drop(columns=ohe_cat_cols).join(df_ohe)
df_encoded.head()

In [ ]:
corr = df_encoded.corr()
corr.style.background_gradient(cmap = 'coolwarm')

In [ ]:
num_imputer = SimpleImputer(strategy='mean')
impute = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
df_encoded[impute] = num_imputer.fit_transform(df_encoded[impute])

In [ ]:
body_mass = df_encoded['body_mass_g'].copy()
mask = np.abs(body_mass - body_mass.mean()) < 3*body_mass.std()
df_encoded = df_encoded[mask]
df_encoded.head()

In [ ]:
scaler = StandardScaler()
scale = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
df_encoded[scale] = scaler.fit_transform(df_encoded[scale])
df_encoded.head()

In [ ]:
X = df_encoded.drop('species', axis=1).copy()
Y = df_encoded['species'].copy()

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X_train, Y_train)

Y_pred = model.predict(X_test)
report = classification_report(Y_test, Y_pred)
print(report)

### AUTOMOBILE DATASET

1. U?itavanje realnog skupa: U?itajte mpg dataset kori?tenjem sns.load_dataset("mpg").

2. Vektorizacija kategorijskih podataka: Kolone kao ?to su origin i name transformisati kori?tenjem OneHotEncoder.

3. Imputacija: Popuniti nedostaju?e vrijednosti u koloni horsepower kori?tenjem srednje vrijednosti. Ukloniti kolone koje imaju visoku korelaciju iznad 0.95.

4. Outlieri: Iz kolone mpg (ciljna varijabla) izbaciti redove sa vrijednostima izvan 3 standardne devijacije.

5. Skaliranje: Primijeniti StandardScaler na numeri?ke kolone.

6. Trening: Podijeliti podatke na trening i test skup, te trenirati DecisionTreeRegressor bez pretrage hiperparametara. Ciljna kolona: mpg.

7. Evaluacija: Evaluirati model kori?tenjem MSE i R? na testnom skupu.

In [ ]:
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split,ParameterGrid
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

In [ ]:
df = sns.load_dataset("mpg")
df.head()

In [ ]:
# provjeravanje u kojim numerickim kolonama ima NaN vrijednosti
num_cols = ['mpg', 'cylinders','displacement','horsepower','weight','acceleration']
df[num_cols].isnull().sum()

In [ ]:
# imputacija podataka samo sa kolonu horsepower jer ona jedina nije potpuna
imputer = SimpleImputer(strategy='mean')
impute = ['horsepower']
df[impute] = imputer.fit_transform(df[impute])


In [ ]:
# Nad klonama origin i name se vrsi LabelEncodovanje - svaki origin dobice jedan index u nizu 
# jedinstvenih pojmova te kolone, isto vazi i za name
le_encode = ['origin','name']
for col in le_encode:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
df.head()

In [ ]:
# racuna se matrica koleracije
corr = df.corr()
corr.style.background_gradient(cmap='coolwarm')

In [ ]:
# uklanjaju se sve kolone koje imaju koleraciju vecu od 95%
df = df.drop('displacement', axis=1)
df.head()

In [ ]:
# nad kategorickim kolonama sada se vrsi OneHotEncodovanje da bi svaki jedinstven pojam dobio svoju kolonu
ohe_cols = ['origin', 'name']
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded_data = ohe.fit_transform(df[ohe_cols])
feature_names = ohe.get_feature_names_out(ohe_cols)
df_ohe = pd.DataFrame(encoded_data, columns=feature_names, index=df.index)
# ohe kodovane kolone se izbacuju is dataseta, a dodaju se njihove rastavljene kolone
df = df.drop(ohe_cols, axis=1).join(df_ohe)
df.head()

In [ ]:
# uklanjaju se outlejeri
mpg = df['mpg'].copy()
mask = np.abs(mpg-mpg.mean()) < 3*mpg.std()
df = df[mask]

In [ ]:
# numericke kolone se skaliraju - sve osim target kolone (mpg)
num_cols = ['cylinders', 'horsepower','weight','acceleration','model_year']
scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])
df.head()

In [ ]:
# podaci se rastavljaju na dva dijela Y - samo target kolona, X - svi preostali podaci
# zatim se dijele u test, validacione i train datasetove, model se trenira sa DecisionTreeRegressorom i AdaBoostRegressorom
# odredjivanje hiperparametara se vrsi tako sto se model testira na vise razlicitih podesavanja i odreedjuju se oni koji daju najbolji rezultat
# zatim se model trenira nad tom kombinacijom i testira
X = df.drop('mpg', axis=1).copy()
Y = df['mpg'].copy()

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=42)
X_val, X_test, Y_val, Y_test = train_test_split(X_test, Y_test, test_size=0.5, random_state=42)

param_grid = {
    'max_depth': [2, 3, 5, 7, None],              # dubina stabla
    'min_samples_leaf': [1, 2, 5, 10],            # minimalni broj uzoraka po listu
    'n_estimators': [50, 100, 200],               # broj slabih modela
    'learning_rate': [0.01, 0.1, 0.5, 1.0]        # brzina u?enja
}


grid = ParameterGrid(param_grid)

best_score = -np.inf
best_params = None

for params in grid:
    tree = DecisionTreeRegressor(max_depth=params['max_depth'],min_samples_leaf=params['min_samples_leaf'])
    
    model = AdaBoostRegressor(
        tree,
        n_estimators=params['n_estimators'],
        learning_rate=params['learning_rate'],
        random_state=42)

    model.fit(X_train, Y_train)
    Y_pred = model.predict(X_val)
    score = r2_score(Y_val, Y_pred)  # kod testiranja hiperparametara koristi se validacioni skup podataka
    print(f"{params} ? R?: {score:.4f}")
    if score > best_score:
        best_score = score
        best_params = params


tree = DecisionTreeRegressor(
        max_depth=best_params['max_depth'],
        min_samples_leaf=best_params['min_samples_leaf']
    )
    
model = AdaBoostRegressor(
        tree,
        n_estimators=best_params['n_estimators'],
        learning_rate=best_params['learning_rate'],
        random_state=42
    )
model.fit(X_train, Y_train)
Y_pred = model.predict(X_test)

In [ ]:
print("MSE: ", mean_squared_error(Y_test,Y_pred))
print("R2S: ", r2_score(Y_test,Y_pred))

### NEURONSKE MREZE

In [ ]:
import torch
from sklearn.datasets import  load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score
import numpy as np

In [ ]:
df = load_iris()

In [ ]:
X, Y = df.data, df.target
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.1, random_state=42, stratify=Y)
X_train, X_val, Y_train, Y_val = train_test_split(X_train, Y_train, test_size=0.2, random_state=42, stratify=Y_train)

In [ ]:
# fit_transform - fit dio racuna srednju vrijednost i standartnu devijaciju, a transform dio standardizuje podatke koristeci ove vrijednosti
# vazno je da se fit_transform ne primijeni na test i na val jer nam ne trebaju ti podaci u trening skupu
# onda nema smisla razdvajati podatke na tri seta
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
X_val = scaler.transform(X_val)

In [ ]:
# provjera tipa podataka za iduci korak
# float64 -> float32
# int64 -> long, prema PyTorch zahtjevima
print(X_train.dtype)
print(X_test.dtype)
print(X_val.dtype)

print(Y_train.dtype)
print(Y_test.dtype)
print(Y_val.dtype)

In [ ]:
# pretvara numpy nizove u PyTorch tenzore (matrice)
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)

Y_train_tensor = torch.tensor(Y_train, dtype=torch.long)
Y_test_tensor = torch.tensor(Y_test, dtype=torch.long)
Y_val_tensor = torch.tensor(Y_val, dtype=torch.long)

In [ ]:
# kreira dataset podatke koji ce se kasnije koristiti zajedno, pakuje X_ i Y_ zajedno u jedan objekat
train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, Y_test_tensor)
val_dataset = TensorDataset(X_val_tensor, Y_val_tensor)

In [ ]:
# kreira male dijelove za preniranje, razdvaja postojece datasetove na vise malih za treniranje
# gdje je batch_size velicina jednog batcha
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)
val_loader = DataLoader(val_dataset, batch_size=16)

In [ ]:
# kreira se mreza za dati dataset - svaka mreza nasljedjuje od nn.Module mreze
# __init__ je konstruktor u kome se definisu 4 ulazne vrijednosti (karakteristike irisa), 
# dva skrivena sloja i ReLU aktivaciona funkcija izmedju svakog sloja 
# na izlazu se nalaze tri neurona (jer imamo 3 klase za iris)
# forward funkcija definise kako se ulazni podaci prenose kroz mrezu
class IrisNet(nn.Module):
    def __init__(self, hidden1, hidden2, dropout_p=0.5):
        super(IrisNet, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(4, hidden1),
            nn.ReLU(),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Linear(hidden2,3)
        )
    
    def forward(self, x):
        return self.net(x)

In [ ]:
# pravimo model koji ce imati 32 i 16 skrivenih slojeva
# definiseno funkciju greske i potimizer koji ima stopu ucenja 0.001
# u zavisnosti od hardvera model ce se trenirati ili na grafickoj kartici ili na procesoru
# model se prebacuje na obradu na dostupan uredjan
model = IrisNet(hidden1=32, hidden2=16)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

In [ ]:
def evaluate(model, loader, device):
    model.eval() # postavlja model u eval rezim
    all_preds, all_labels = [], [] # lista za predikcije i stvarne vrijednosti
    with torch.inference_mode(): # ne racuna gradijent
        for X_batch, Y_batch in loader:
            X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)
            outputs = model(X_batch) # vraca neskalirane vrijednosti - logits
            probabilities = torch.softmax(outputs, 1) # pretvara logits u vjerovatnoce po klasi
            _ , preds= torch.max(probabilities, 1) # vraca najvecu vjerovatnocu po redu
            all_preds.extend(preds.cpu().numpy())   # prebacivanje tensora na cpu i u numpy format
            all_labels.extend(Y_batch.cpu().numpy()) # - || -

    return accuracy_score(all_labels, all_preds)

In [ ]:
evaluate(model, test_loader, device)

In [ ]:
best_val_loss = np.inf
best_val_acc = 0
epoch_no_improve = 0
n_epochs = 50
patience = 5

for epoch in range(n_epochs):
    model.train()
    train_loss = 0
    for X_batch, Y_batch in train_loader:
        X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)
        optimizer.zero_grad() # resetovanje gradijenata
        outputs = model(X_batch) # izlaz modela
        loss = criterion(outputs, Y_batch) # proracun gubitka
        loss.backward() # propagacija unazad za racunanje gradijenta
        optimizer.step() # azuriranje tezine optimizerom
        train_loss = loss.item() * X_batch.size(0)
    train_loss/= len(train_loader.dataset) # ukupan trening gubitak

    model.eval()
    val_loss = 0.0
    with torch.inference_mode():
        for X_batch, Y_batch in val_loader:
            X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)
            outputs = model(X_batch)
            val_loss += criterion(outputs, Y_batch).item() * X_batch.size(0)
        # racuna gubitak na validacionom skupu podataka
        val_loss /= len(val_loader.dataset)
        val_acc = evaluate(model, val_loader, device)

        # ako je validaciona greska bolja od prethodne najbolje podaci se azuriraju i model se cuva
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            epoch_no_improve = 0
            torch.save(model.state_dict(), 'best_iris_model.pth')
        else:
            epoch_no_improve +=1 # inace broj iteracija bez napretka se povecava

        # i kada dostigne maksimum nivoa koje ignorisemo trening modela se 
        # zaustavlja jer vise nema smisla trenirati ga jer ne uci nista novo
        if epoch_no_improve>=patience:
            print("Early stopping triggered.")
            break

In [ ]:
# ucitavanje modela i testiranje na testom skupu podataka
model.load_state_dict(torch.load('best_iris_model.pth'))
test_acc = evaluate(model, test_loader, device)
print(test_acc)

output = model(X_test_tensor.to(device))
probabilities = torch.softmax(output,1)
print(probabilities)

### VJEZBA - treci lab sa pretragom hiperparametara

1. **U?itavanje realnog skupa**: U?itajte tips skup podataka upotrebom sns biblioteke: https://rdrr.io/cran/reshape2/man/tips.html
2. **Vektorizacija kategorickih podataka**: Identifikovati kategori?ke kolone. Na odgovaraju?im kolonama primjeniti LabelEncoder i OneHotEncoder
3. **Imputacija**: Detektovati da li postoje NaN vrijednosti te ih rje?iti odgovaraju?om tehnikom. Detektovati kolone koje posjeduju visok stepen korelacije te izbrisati odgovaraju?e kolone.
4. **Outlieri**: Definisati prag od 3 standardne devijacije za numeri?ke kolonu total_bill te izvr?iti uklanjanje outlier-a.
5. **Skaliranje**: Primijeniti `StandardScaler` ili `MinMaxScaler` na total_bill koloni.
6. **Trening**: Podijeliti podatke na training/test i istrenirati DecisionTree za regresiju koji koristi AdaBoost algoritam za trening u ansamblu. Kao ciljnu kolonu koristiti tip.
7. **Evaluacija**: Evaluirati metrike na testnom skupu podataka.

In [ ]:
import seaborn as sns

data = sns.load_dataset('tips')
data.head()

In [ ]:
data.isnull().sum()

In [ ]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
import pandas as pd
data_encoded = data.copy()

categorical_columns = ['sex', 'smoker', 'day', 'time']
label_encoders = {}

for column in categorical_columns:
    le = LabelEncoder()
    le.fit(data[column])
    data_encoded[column] = le.fit_transform(data[column])
    label_encoders[column] = le

data_encoded.head()

In [ ]:
corr = data_encoded.corr()
corr.style.background_gradient(cmap='coolwarm')

In [ ]:
ohe_cat_cols = ['day','time']
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded_data = ohe.fit_transform(data_encoded[ohe_cat_cols])
feature_names = ohe.get_feature_names_out(ohe_cat_cols)
data_ohe = pd.DataFrame(encoded_data, columns=feature_names, index=data.index)
data_encoded=data_encoded.drop(ohe_cat_cols, axis=1).join(data_ohe)
data_encoded.head()

In [ ]:
import numpy as np
total_bill = data_encoded['total_bill'].copy()
mask = np.abs(total_bill - total_bill.mean()) < 3*total_bill.std()
data_encoded = data_encoded[mask]
data_encoded.head()

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
data_encoded['total_bill'] = scaler.fit_transform(data_encoded[['total_bill']])
data_encoded.head()

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.metrics import mean_absolute_error

X = data_encoded.drop('tip',axis=1).to_numpy()
Y = data_encoded['tip'].to_numpy()

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
X_train, X_val, Y_train, Y_val = train_test_split(X_train, Y_train, test_size=0.2, random_state=42)

param_grid = {
    'criterion': ['squared_error','friedman_mse', 'poisson'],
    'max_depth': [None, 5, 10],
    'min_samples_leaf': [1, 2, 4, 8],
    'min_samples_split': [2, 4, 8]
}

grid = ParameterGrid(param_grid)

best_score = np.inf
best_params = None

for params in grid:
    model = DecisionTreeRegressor(**params)
    model.fit(X_train, Y_train)
    Y_pred = model.predict(X_val)
    score = mean_absolute_error(Y_pred, Y_val)

    if score < best_score:
        best_score, best_params = score, params

print("Best score:", best_score)
print("Best params: ", best_params)

In [ ]:
model = DecisionTreeRegressor(**best_params)
model.fit(X_train, Y_train)
Y_pred = model.predict(X_test)

print("MAE: ", mean_absolute_error(Y_pred, Y_test))

###  Adult Dataset

In [ ]:
import pandas as pd

url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'

column_names = [
    'age', 'workclass', 'fnlwgt', 'education', 'education_num',
    'marital_status', 'occupation', 'relationship', 'race', 'sex',
    'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income'
]
df = pd.read_csv(url, names = column_names, na_values='?', skipinitialspace=True)
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='most_frequent')
impute = ['workclass', 'occupation', 'native_country']

df[impute] = imputer.fit_transform(df[impute])
df.head()

In [ ]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

label_cols = ['income','workclass','education', 'marital_status', 'occupation', 'relationship', 'race', 'native_country', 'sex', ]
label_encoders = {}
df_encoded = df.copy()
for col in label_cols:
    le = LabelEncoder()
    le.fit(df_encoded[col])
    df_encoded[col] = le.fit_transform(df_encoded[col])
    label_encoders[col] = le

In [ ]:
corr = df_encoded.corr()
corr.style.background_gradient(cmap='coolwarm')

In [ ]:
drop = ['relationship']
df_encoded = df_encoded.drop(columns=drop)

In [ ]:
ohe_cols = ['workclass','education', 'marital_status', 'occupation', 'race', 'native_country', 'sex']
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded_data = ohe.fit_transform(df_encoded[ohe_cols])
feature_names = ohe.get_feature_names_out(ohe_cols)
df_ohe = pd.DataFrame(encoded_data, columns=feature_names, index = df.index)
df_encoded = df_encoded.drop(columns=ohe_cols).join(df_ohe)
df_encoded.head()

In [ ]:
from sklearn.preprocessing import StandardScaler

scale = ['age', 'fnlwgt','education_num', 'capital_gain', 'capital_loss', 'hours_per_week']
scaler = StandardScaler()
df_encoded[scale] = scaler.fit_transform(df_encoded[scale])
df_encoded.head()

In [ ]:
from sklearn.model_selection import train_test_split

X = df_encoded.drop('income', axis=1).to_numpy()
Y = df_encoded['income'].to_numpy()

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2)
X_train, X_val, Y_train, Y_val = train_test_split(X_train, Y_train, test_size=0.2)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import mean_absolute_error
import numpy as np

param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}


grid = ParameterGrid(param_grid)
best_score = np.inf
best_params = None

for params in grid:
    model = DecisionTreeClassifier(**params)
    model.fit(X_train, Y_train)
    Y_pred = model.predict(X_val)
    score = mean_absolute_error(Y_pred, Y_val)

    if score<best_score:
        best_score = score
        best_params = params

print("Best score:", best_score)
print("Best params: ", best_params)

model = DecisionTreeClassifier(**best_params)
model.fit(X_train, Y_train)
Y_pred = model.predict(X_test)

print("MAE: ",mean_absolute_error(Y_pred, Y_test))